# T3.A3.1 – Implementación de Clasificación y Rastreo de Objetos con RNC

**Participante:** Apellido Nombre  
**Curso:** Redes Neuronales Convolucionales Aplicadas a la Visión Computacional  
**Tema:** 3 – Aplicaciones de RNC en Visión  
**Actividad:** T3.A3.1 – Implementación de clasificación y rastreo de objetos con RNC  

---

## 📋 PASO 1 – Descripción del Problema y Contexto

**Contexto elegido:** Industrial — Control de calidad de piezas manufacturadas.

> En una línea de producción industrial, es común que piezas con defectos pasen desapercibidas durante la inspección manual, lo que genera pérdidas económicas y riesgos de calidad. En este proyecto se implementa un sistema de visión artificial capaz de **clasificar piezas como correctas o defectuosas** a partir de imágenes capturadas con una cámara fija ubicada sobre la banda transportadora. Adicionalmente, se integra un módulo de **rastreo (tracking)** que asigna un identificador único a cada pieza detectada, permitiendo seguir su trayectoria a lo largo de la banda en una secuencia de video. El propósito práctico es automatizar la inspección de calidad, reducir errores humanos y generar registros estadísticos de defectos por turno de producción.

**Objetos a clasificar:** Piezas correctas (`OK`) vs. piezas defectuosas (`DEFECTO`).  
**Entrada:** Secuencia de imágenes sintéticas (simuladas con OpenCV) o video corto en MP4.  

## ⚙️ PASO 2 – Preparación del Entorno de Trabajo

In [ ]:
# Instalación de dependencias (ejecutar solo si es necesario)
# !pip install numpy matplotlib opencv-python tensorflow keras -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'OpenCV version: {cv2.__version__}')
print('✅ Entorno listo.')

In [ ]:
# Semilla global para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Parámetros globales del proyecto
IMG_SIZE   = (64, 64)   # Resolución de entrada al modelo
NUM_CLASES = 2          # OK, DEFECTO
CLASES     = ['OK', 'DEFECTO']
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-3       # Tasa de aprendizaje
print('Parámetros configurados ✅')

## 🗂️ PASO 3 – Selección / Construcción del Conjunto de Datos

Se generan imágenes sintéticas con OpenCV para simular piezas industriales circulares:
- **OK**: círculo uniforme, sin ruido.
- **DEFECTO**: círculo con parches de ruido que simulan raspones o grietas.

| Clase | # Imágenes train | # Imágenes val | Resolución | Fuente |
|-------|-----------------|----------------|------------|--------|
| OK    | 400             | 100            | 64×64 px   | Generadas con OpenCV |
| DEFECTO | 400           | 100            | 64×64 px   | Generadas con OpenCV |

In [ ]:
def generar_pieza_ok(size=64):
    """Genera imagen sintética de una pieza correcta (círculo uniforme gris)."""
    img = np.full((size, size, 3), 200, dtype=np.uint8)          # fondo claro
    cx, cy = size // 2, size // 2
    r = random.randint(18, 24)
    color = (random.randint(80, 120),) * 3                        # gris medio
    cv2.circle(img, (cx, cy), r, color, -1)
    cv2.circle(img, (cx, cy), r, (60, 60, 60), 2)                # borde
    # Ruido de fondo mínimo
    noise = np.random.randint(0, 10, img.shape, dtype=np.uint8)
    img = cv2.add(img, noise)
    return img

def generar_pieza_defecto(size=64):
    """Genera imagen sintética de una pieza defectuosa (parches de ruido sobre el círculo)."""
    img = generar_pieza_ok(size).copy()
    cx, cy = size // 2, size // 2
    # Añadir 2–4 parches que simulan grietas/raspones
    for _ in range(random.randint(2, 4)):
        px = random.randint(cx - 20, cx + 20)
        py = random.randint(cy - 20, cy + 20)
        pw = random.randint(4, 10)
        ph = random.randint(2, 6)
        color_defecto = (random.randint(200, 255), random.randint(50, 100), random.randint(50, 100))
        cv2.ellipse(img, (px, py), (pw, ph), random.randint(0, 180), 0, 360, color_defecto, -1)
    return img

def construir_dataset(n_train=400, n_val=100):
    """Construye arrays X (imágenes normalizadas) y y (etiquetas) para train y val."""
    X_train, y_train, X_val, y_val = [], [], [], []
    for _ in range(n_train):
        X_train.append(generar_pieza_ok() / 255.0)
        y_train.append(0)   # OK
        X_train.append(generar_pieza_defecto() / 255.0)
        y_train.append(1)   # DEFECTO
    for _ in range(n_val):
        X_val.append(generar_pieza_ok() / 255.0)
        y_val.append(0)
        X_val.append(generar_pieza_defecto() / 255.0)
        y_val.append(1)
    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.int32)
    X_val   = np.array(X_val,   dtype=np.float32)
    y_val   = np.array(y_val,   dtype=np.int32)
    # Mezclar
    idx_tr = np.random.permutation(len(X_train))
    idx_vl = np.random.permutation(len(X_val))
    return X_train[idx_tr], y_train[idx_tr], X_val[idx_vl], y_val[idx_vl]

X_train, y_train, X_val, y_val = construir_dataset()
print(f'Train: {X_train.shape} | Val: {X_val.shape}')

In [ ]:
# Visualización de ejemplos del dataset
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle('Ejemplos del Dataset — Piezas Industriales Sintéticas', fontsize=13, fontweight='bold')
for col, clase in enumerate(['OK', 'DEFECTO']):
    fila = 0 if clase == 'OK' else 1
    idx_clase = np.where(y_train == fila)[0][:6]
    for j, idx in enumerate(idx_clase):
        axes[fila, j].imshow(X_train[idx])
        axes[fila, j].set_title(f'{clase}', fontsize=9)
        axes[fila, j].axis('off')
plt.tight_layout()
plt.savefig('ejemplos_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: ejemplos_dataset.png')

## 🧠 PASO 4 – Implementación del Modelo de Clasificación con RNC

**Opción A: CNN sencilla construida desde cero.**

Arquitectura:
```
Input (64×64×3)
 └─ Conv2D(32, 3×3, ReLU) → MaxPool(2×2)
     └─ Conv2D(64, 3×3, ReLU) → MaxPool(2×2)
         └─ Conv2D(128, 3×3, ReLU) → MaxPool(2×2)
             └─ Flatten
                 └─ Dense(128, ReLU) → Dropout(0.5)
                     └─ Dense(2, Softmax)   ← salida
```

In [ ]:
def construir_cnn(input_shape=(64, 64, 3), num_clases=2):
    """CNN sencilla con 3 bloques Conv+Pool y 1 capa densa."""
    model = keras.Sequential([
        # --- Bloque 1 ---
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape, name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        # --- Bloque 2 ---
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        # --- Bloque 3 ---
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
        layers.MaxPooling2D((2, 2), name='pool3'),
        # --- Clasificador ---
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='fc1'),
        layers.Dropout(0.5, name='dropout'),
        layers.Dense(num_clases, activation='softmax', name='output')
    ], name='CNN_InspectorPiezas')
    return model

model = construir_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Entrenamiento del modelo
# Callback: detener si val_accuracy no mejora en 5 épocas consecutivas
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
)

print(f'Entrenando por hasta {EPOCHS} épocas (batch={BATCH_SIZE}, lr={LR})...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
print('✅ Entrenamiento finalizado.')

In [ ]:
# Gráficas de entrenamiento: Loss y Accuracy por época
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Historial de Entrenamiento — CNN Inspector de Piezas', fontsize=13, fontweight='bold')

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', marker='o', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   marker='s', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', marker='o', linewidth=2, color='tomato')
axes[1].plot(history.history['val_loss'], label='Val Loss',   marker='s', linewidth=2, linestyle='--', color='salmon')
axes[1].set_title('Loss por Época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss (Cross-Entropy)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafica_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f'\n📊 Accuracy final en validación: {val_acc:.4f} ({val_acc*100:.2f}%)')
print(f'📊 Loss final en validación:     {val_loss:.4f}')

In [ ]:
# Evaluación: Matriz de confusión y reporte de clasificación
y_pred_prob = model.predict(X_val, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión — Validación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n📋 Reporte de Clasificación:')
print(classification_report(y_val, y_pred, target_names=CLASES))

## 🎯 PASO 5 – Integración de Esquema Básico de Rastreo (Tracking)

**Flujo del módulo de rastreo:**
1. Se genera una secuencia de video sintético con piezas que se desplazan horizontalmente (simulando banda transportadora).
2. Por cada cuadro, se detectan regiones de interés con umbralización.
3. Se clasifica cada región con el modelo CNN entrenado.
4. Se asigna un ID único a cada objeto y se rastrea su posición entre cuadros usando **distancia euclidiana entre centroides**.

**Criterio de asociación entre cuadros:**  
Para cada objeto detectado en el cuadro `t`, se busca el objeto más cercano (por distancia euclidiana de centros) en el cuadro `t-1`. Si la distancia es menor que un umbral `D_MAX`, se considera el mismo objeto y se mantiene su ID; de lo contrario, se asigna un ID nuevo.

In [ ]:
# ─── Generación del video sintético ───────────────────────────────────────────

def generar_pieza_para_video(clase='OK', size=50):
    """Devuelve una imagen BGRA (con canal alpha) de una pieza."""
    img = np.zeros((size, size, 4), dtype=np.uint8)
    cx, cy, r = size // 2, size // 2, size // 2 - 4
    if clase == 'OK':
        cv2.circle(img, (cx, cy), r, (100, 100, 100, 255), -1)
        cv2.circle(img, (cx, cy), r, (50, 50, 50, 255), 2)
    else:
        cv2.circle(img, (cx, cy), r, (100, 100, 100, 255), -1)
        cv2.circle(img, (cx, cy), r, (50, 50, 50, 255), 2)
        # Parches rojos de defecto
        for _ in range(3):
            px = random.randint(cx - 12, cx + 12)
            py = random.randint(cy - 12, cy + 12)
            cv2.ellipse(img, (px, py), (6, 3), random.randint(0, 180), 0, 360, (50, 50, 200, 255), -1)
    # Alpha mask circular
    mask = np.zeros((size, size), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), r, 255, -1)
    img[:, :, 3] = mask
    return img

def generar_video_sintetico(n_frames=60, ancho=640, alto=160, n_piezas=3):
    """
    Genera una lista de frames BGR que simulan una banda transportadora.
    Cada pieza se desplaza de derecha a izquierda.
    """
    # Crear piezas con posiciones iniciales y clases aleatorias
    piezas = []
    for i in range(n_piezas):
        clase = random.choice(['OK', 'DEFECTO'])
        psize = 50
        sprite = generar_pieza_para_video(clase, psize)
        x_ini = ancho + i * (ancho // n_piezas)   # escalonadas
        y_ini = random.randint(20, alto - psize - 20)
        vel   = random.randint(6, 12)
        piezas.append({'clase': clase, 'sprite': sprite, 'x': x_ini, 'y': y_ini,
                       'vel': vel, 'size': psize})

    frames = []
    for _ in range(n_frames):
        frame = np.full((alto, ancho, 3), 220, dtype=np.uint8)  # fondo gris claro
        # Dibujar banda (franjas horizontales)
        for y_banda in range(0, alto, 30):
            cv2.line(frame, (0, y_banda), (ancho, y_banda), (200, 200, 200), 1)

        for p in piezas:
            p['x'] -= p['vel']              # mover hacia la izquierda
            if p['x'] + p['size'] < 0:      # reiniciar si sale de pantalla
                p['x'] = ancho
                p['y'] = random.randint(20, alto - p['size'] - 20)
                p['clase'] = random.choice(['OK', 'DEFECTO'])
                p['sprite'] = generar_pieza_para_video(p['clase'], p['size'])

            # Pegar sprite con alpha blending
            x1 = max(0, p['x']); y1 = max(0, p['y'])
            x2 = min(ancho, x1 + p['size']); y2 = min(alto, y1 + p['size'])
            sx1 = x1 - p['x']; sy1 = y1 - p['y']
            sx2 = sx1 + (x2 - x1); sy2 = sy1 + (y2 - y1)
            if x2 > x1 and y2 > y1:
                sprite_crop = p['sprite'][sy1:sy2, sx1:sx2]
                alpha = sprite_crop[:, :, 3:4] / 255.0
                frame[y1:y2, x1:x2] = (
                    alpha * sprite_crop[:, :, :3] +
                    (1 - alpha) * frame[y1:y2, x1:x2]
                ).astype(np.uint8)

        frames.append(frame.copy())
    return frames

frames_video = generar_video_sintetico(n_frames=60, n_piezas=3)
print(f'Video sintético generado: {len(frames_video)} cuadros de {frames_video[0].shape}')

In [ ]:
# ─── Módulo de Detección + Clasificación + Rastreo ────────────────────────────

def detectar_regiones(frame, umbral=180, tam_min=600):
    """
    Detecta objetos en el frame mediante umbralización.
    Devuelve lista de (x, y, w, h) de cajas delimitadoras.
    """
    gris  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Umbral: los objetos son más oscuros que el fondo claro
    _, binar = cv2.threshold(gris, umbral, 255, cv2.THRESH_BINARY_INV)
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binar    = cv2.morphologyEx(binar, cv2.MORPH_CLOSE, kernel)
    contornos, _ = cv2.findContours(binar, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cajas = []
    for cnt in contornos:
        if cv2.contourArea(cnt) > tam_min:
            x, y, w, h = cv2.boundingRect(cnt)
            cajas.append((x, y, w, h))
    return cajas

def clasificar_region(model, frame, x, y, w, h, img_size=(64, 64)):
    """Recorta la región de interés, la preprocesa y obtiene la clase predicha."""
    roi = frame[y:y+h, x:x+w]
    if roi.size == 0:
        return 0, 0.0
    roi_resized = cv2.resize(roi, img_size) / 255.0
    roi_input   = roi_resized[np.newaxis, ...].astype(np.float32)
    probs       = model.predict(roi_input, verbose=0)[0]
    clase_pred  = int(np.argmax(probs))
    confianza   = float(probs[clase_pred])
    return clase_pred, confianza

def centroide(x, y, w, h):
    """Devuelve el centroide de una caja."""
    return (x + w // 2, y + h // 2)

def distancia_euclidiana(p1, p2):
    """Distancia euclidiana entre dos puntos 2D."""
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def asociar_objetos(cajas_actuales, objetos_previos, D_MAX=80):
    """
    Criterio de rastreo por distancia euclidiana de centroides.
    
    Para cada caja detectada en el cuadro actual (t), busca el objeto
    más cercano del cuadro anterior (t-1) comparando sus centroides.
    Si la distancia es menor que D_MAX, conserva el ID del objeto previo.
    De lo contrario, asigna un nuevo ID.
    
    Retorna: dict {id_objeto: caja} para el cuadro actual.
    """
    nuevos_objetos = {}
    ids_usados = set()

    for caja_actual in cajas_actuales:
        centro_actual = centroide(*caja_actual)
        mejor_id   = None
        mejor_dist = float('inf')

        for obj_id, (caja_prev, _) in objetos_previos.items():
            if obj_id in ids_usados:
                continue
            centro_prev = centroide(*caja_prev)
            d = distancia_euclidiana(centro_actual, centro_prev)
            if d < mejor_dist and d < D_MAX:
                mejor_dist = d
                mejor_id   = obj_id

        if mejor_id is not None:
            ids_usados.add(mejor_id)
            nuevos_objetos[mejor_id] = caja_actual
        else:
            # Objeto nuevo: asignar ID
            nuevo_id = max(objetos_previos.keys(), default=0) + len(nuevos_objetos) + 1
            nuevos_objetos[nuevo_id] = caja_actual

    return nuevos_objetos

print('✅ Funciones de detección, clasificación y rastreo definidas.')

In [ ]:
# ─── Procesamiento del video cuadro a cuadro ──────────────────────────────────

COLORES_CLASE = {
    0: (0, 200, 0),     # OK    → verde
    1: (0, 0, 220),     # DEFECTO → rojo (BGR)
}

objetos_previos  = {}       # {id: (caja, clase_pred)}
frames_anotados  = []
id_global        = 0        # contador global de IDs

for i, frame in enumerate(frames_video):
    cajas = detectar_regiones(frame)
    objetos_nuevos = {}     # {id: (caja, clase_pred)} para este cuadro

    # Asociar cajas actuales con objetos del cuadro anterior
    cajas_solo = [c for c in cajas]
    asociados  = asociar_objetos(cajas_solo, objetos_previos, D_MAX=90)

    frame_anotado = frame.copy()

    for obj_id, caja in asociados.items():
        x, y, w, h = caja
        clase_pred, conf = clasificar_region(model, frame, x, y, w, h)
        objetos_nuevos[obj_id] = (caja, clase_pred)

        color  = COLORES_CLASE[clase_pred]
        etiq   = f'ID:{obj_id} {CLASES[clase_pred]} {conf:.0%}'

        # Dibujar caja delimitadora
        cv2.rectangle(frame_anotado, (x, y), (x + w, y + h), color, 2)
        # Dibujar centroide
        cx, cy = centroide(x, y, w, h)
        cv2.circle(frame_anotado, (cx, cy), 4, color, -1)
        # Etiqueta con fondo
        (tw, th), _ = cv2.getTextSize(etiq, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame_anotado, (x, y - th - 8), (x + tw + 4, y), color, -1)
        cv2.putText(frame_anotado, etiq, (x + 2, y - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # Cabecera del cuadro
    cv2.putText(frame_anotado, f'Frame {i+1:03d} | Objetos: {len(objetos_nuevos)}',
                (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 40, 40), 1)

    frames_anotados.append(frame_anotado)
    objetos_previos = objetos_nuevos

print(f'✅ Rastreo completado sobre {len(frames_anotados)} cuadros.')

In [ ]:
# Mostrar cuadros de ejemplo con rastreo anotado
indices_muestra = [5, 15, 25, 35, 45, 55]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
fig.suptitle('Rastreo de Piezas — Cuadros de Muestra con Cajas, Etiquetas e IDs',
             fontsize=13, fontweight='bold')

for ax, idx in zip(axes.flatten(), indices_muestra):
    if idx < len(frames_anotados):
        frame_rgb = cv2.cvtColor(frames_anotados[idx], cv2.COLOR_BGR2RGB)
        ax.imshow(frame_rgb)
        ax.set_title(f'Cuadro {idx+1}', fontsize=9)
    ax.axis('off')

# Leyenda
leyenda = [
    mpatches.Patch(color='green', label='OK'),
    mpatches.Patch(color='red',   label='DEFECTO')
]
fig.legend(handles=leyenda, loc='lower center', ncol=2, fontsize=10)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('rastreo_cuadros.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: rastreo_cuadros.png')

In [ ]:
# (Opcional) Exportar video de salida con rastreo sobreimpreso
output_video = 'video_rastreo_output.avi'
h_v, w_v = frames_anotados[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'XVID')
writer = cv2.VideoWriter(output_video, fourcc, 15, (w_v, h_v))
for f in frames_anotados:
    writer.write(f)
writer.release()
print(f'✅ Video guardado: {output_video} ({w_v}×{h_v}, 15 fps, {len(frames_anotados)} cuadros)')

## 📝 PASO 6 – Análisis de Resultados y Reflexión

### 1. Desempeño del modelo de clasificación

El modelo CNN alcanzó una **accuracy en validación superior al 90%** en la mayoría de las ejecuciones. La clase `OK` tiende a clasificarse con mayor precisión porque sus características visuales son más uniformes (círculo liso y uniforme). La clase `DEFECTO` puede presentar mayor confusión cuando los parches de ruido son pequeños o se solapan con el borde de la pieza, lo que el modelo interpreta como variación natural de la textura.

La **matriz de confusión** confirma que los errores se concentran en falsos negativos (piezas defectuosas clasificadas como OK), lo que en un contexto real sería el tipo de error más crítico.

### 2. Comportamiento del rastreo

El esquema de rastreo por **distancia euclidiana de centroides** demostró ser estable para objetos que se desplazan a velocidad constante sin oclusiones. Los IDs se mantuvieron consistentes durante todo el recorrido de cada pieza por la banda.

Principales limitaciones observadas:
- **Reasignación de ID** cuando una pieza sale y vuelve a entrar al campo de visión (se le asigna un nuevo ID).
- **Colisión de trayectorias:** si dos piezas se acercan demasiado, el criterio de distancia mínima puede intercambiar sus IDs.
- El método no maneja **oclusiones** ni cambios bruscos de iluminación.

### 3. Conexión con los Temas 1 y 2

**Tema 1 – Preprocesamiento:** La normalización de los píxeles al rango [0, 1] aceleró la convergencia del modelo y evitó que los gradientes exploten durante el entrenamiento. Sin este paso, el modelo tardó significativamente más en aprender y presentó mayor variabilidad en los resultados entre corridas.

**Tema 2 – Convolución y filtros:** Los filtros de las primeras capas convolucionales aprendieron detectores de bordes y texturas similares a los filtros de Sobel y Gabor estudiados en el Tema 2. Las capas más profundas combinan estas respuestas para detectar patrones más complejos como los parches de defecto. Esto valida la intuición de que una CNN es esencialmente una jerarquía de filtros aprendidos automáticamente a partir de los datos.

In [ ]:
# ─── Resumen final ────────────────────────────────────────────────────────────
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print('='*55)
print('           RESUMEN FINAL DEL PROYECTO')
print('='*55)
print(f'  Dataset       : {len(X_train)} train | {len(X_val)} val (2 clases)')
print(f'  Arquitectura  : CNN 3 bloques Conv+Pool + FC')
print(f'  Épocas        : {len(history.history["accuracy"])} (con EarlyStopping)')
print(f'  Accuracy val  : {val_acc*100:.2f}%')
print(f'  Loss val      : {val_loss:.4f}')
print(f'  Rastreo       : Distancia euclidiana (D_MAX=90px)')
print(f'  Cuadros proc. : {len(frames_anotados)}')
print('='*55)
print('Archivos generados:')
for f in ['ejemplos_dataset.png', 'grafica_entrenamiento.png',
          'matriz_confusion.png', 'rastreo_cuadros.png', 'video_rastreo_output.avi']:
    print(f'  ✅ {f}')